# Análisis de vuelos
_____
# Limpieza y transformación

En este proyecto se va a proceder con la limpieza del dataset obtenido en la primer etapa.
También se procede a la ingeniería de features para poder realizar reportes con valores como hora, nes.

## Constantes

In [1]:
MAPEO_COLUMNAS = {
    "fecha": "fecha",
    "Hora Programada": "hora_programada",
    "Demora en despegar": "demora_cruda",
}

ETIQUETAS_NIVEL_DEMORA = [
    "A tiempo/Adelantado",
    "Demora Leve (0-15m)",
    "Demora Media (15-45m)",
    "Demora Grave (45m-3h)",
    "Crítico (>3h)"
]

ORDEN_NIVEL_DEMORA = ETIQUETAS_NIVEL_DEMORA + ["Cancelado"]

BINS_DEMORA = [-float("inf"), 0, 15, 45, 180, float("inf")]

PALABRAS_CLAVE_DEMORA = {
    "cancelado": ["cancelado"],
    "a_tiempo": ["a tiempo"],
    "adelantado": ["adelantado"],
    "tarde": ["tarde"],
}
COLUMNAS_CRUDAS_REQUERIDAS = list(MAPEO_COLUMNAS.keys())


## Funciones auxiliares

In [2]:
import io
import os
from typing import Optional

import pandas as pd


def cargar_datos(file_path, **kwargs):
    """
    Carga un archivo CSV o Parquet en un DataFrame de pandas.

    :param file_path: Ruta al archivo.
    :param kwargs: Argumentos adicionales para pd.read_csv / pd.read_parquet.
    :return: DataFrame cargado.
    :raises ValueError: Si la extensión del archivo no es .csv ni .parquet.
    :raises RuntimeError: Si el archivo no se puede leer por otro motivo.
    """
    _, extension_archivo = os.path.splitext(file_path)
    extension_archivo = extension_archivo.lower()

    if extension_archivo == ".csv":
        lector = pd.read_csv
    elif extension_archivo == ".parquet":
        lector = pd.read_parquet
    else:
        raise ValueError(f"Formato no soportado: {extension_archivo}. Usá CSV o Parquet.")

    try:
        return lector(file_path, **kwargs)
    except Exception as e:
        raise RuntimeError(f"Error al cargar el archivo '{file_path}': {e}") from e


def validar_columnas_requeridas(df: pd.DataFrame, columnas_requeridas: list[str]) -> None:
    """
    Lanza un error claro si al DataFrame le falta alguna columna esperada,
    en vez de fallar más adelante con un KeyError confuso en medio del pipeline.
    """
    faltantes = [col for col in columnas_requeridas if col not in df.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas esperadas: {faltantes}. Columnas encontradas: {list(df.columns)}")


def imprimir_reporte(titulo: str, data) -> None:
    """Imprime un bloque de texto/datos con un encabezado simple tipo ASCII."""
    ancho = max(len(titulo), 25)
    print("=" * 40)
    print(titulo.upper())
    print("-" * ancho)
    print(data)
    print("=" * 40)


def reporte_basico_df(df: pd.DataFrame, titulo: str) -> None:
    """Imprime un resumen rápido de un DataFrame: shape, info, describe, nulos y únicos."""
    print("=" * 40)
    print("Reporte:\t", titulo)

    imprimir_reporte("Shape:", f"Columnas: {df.shape[1]}\t|\tFilas: {df.shape[0]}")

    # df.info() imprime directamente y devuelve None, así que lo capturamos
    # en un buffer en vez de pasarle su valor de retorno (vacío) a imprimir_reporte.
    buffer_info = io.StringIO()
    df.info(buf=buffer_info)
    imprimir_reporte("Info", buffer_info.getvalue())

    imprimir_df(df.describe().T)

    imprimir_reporte("Cantidad de valores nulos:", df.isnull().sum())

    imprimir_reporte("Valores únicos:", df.nunique())


def imprimir_df(df_desc: pd.DataFrame) -> None:
    """Imprime un DataFrame con formato usando tabulate (requiere `pip install tabulate`)."""
    from tabulate import tabulate

    # 'tablefmt' puede ser "grid", "fancy_grid", "pipe" o "psql"
    print(tabulate(df_desc, headers="keys", tablefmt="psql"))


## Transformación

In [3]:
import re

def construir_fecha_hora_programada(
    df: pd.DataFrame,
    col_fecha: str,
    col_hora: str,
    col_destino: str,
) -> pd.DataFrame:
    """
    Combina una columna de fecha y una columna de hora programada (texto)
    en una única columna datetime.

    :param col_fecha: Columna con la fecha del vuelo.
    :param col_hora: Columna con la hora programada (texto).
    :param col_destino: Nombre de la nueva columna datetime a crear.
    """
    df[col_destino] = pd.to_datetime(
        df[col_fecha].astype(str) + " " + df[col_hora].astype(str),
        errors="coerce",
    )
    return df


def parsear_demora(valor, palabras_clave) -> tuple[Optional[int], str]:
    """
    Parsea un texto crudo de demora (en español, ej: "Demora 1hs 20min",
    "Cancelado", "A tiempo") y devuelve una tupla (minutos_demora, estado).

    :return: (minutos, estado). minutos es None para vuelos cancelados,
        negativo para vuelos adelantados, 0 para vuelos a tiempo, y positivo
        para vuelos demorados.
    """
    texto = str(valor).lower().strip()

    if any(p in texto for p in palabras_clave["cancelado"]):
        return None, "Cancelado"
    if any(p in texto for p in palabras_clave["a_tiempo"]):
        return 0, "A tiempo"

    match_horas = re.search(r"(\d+)\s*hs?", texto)
    match_minutos = re.search(r"(\d+)\s*min", texto)
    total_minutos = (int(match_horas.group(1)) * 60 if match_horas else 0) + (
        int(match_minutos.group(1)) if match_minutos else 0
    )

    if any(p in texto for p in palabras_clave["adelantado"]):
        return -total_minutos, "Adelantado"
    if any(p in texto for p in palabras_clave["tarde"]) or total_minutos > 0:
        return total_minutos, "Demorado"

    return 0, "A tiempo"


def calcular_demoras(df, col_origen, col_demora="minutos_netos_demora", col_estado="estado"):
    """Parsea la columna de texto crudo de demora en columnas de minutos + estado."""
    df[[col_demora, col_estado]] = df[col_origen].apply(
        lambda x: pd.Series(parsear_demora(x, PALABRAS_CLAVE_DEMORA))
    )
    return df

def procesar_vuelos(df, col_demora_cruda="demora_cruda", col_fecha="fecha", col_hora="hora_programada"):
    """
    Corre el pipeline completo de limpieza + ingeniería de features sobre un
    DataFrame crudo de vuelos: parsea demoras, marca cancelaciones, construye
    la fecha-hora programada, extrae la hora del día y categoriza las
    demoras en niveles.
    """
    validar_columnas_requeridas(df, [col_demora_cruda, col_fecha, col_hora])

    df[col_fecha] = pd.to_datetime(df[col_fecha])

    # 1. Parsear el texto de demora en minutos + estado
    df = calcular_demoras(df, col_demora_cruda, col_demora="minutos_netos_demora", col_estado="estado")

    # 2. Flag de cancelación
    df["esta_cancelado"] = df[col_demora_cruda].str.contains("Cancelado", na=False, case=False)

    # 3. Fecha-hora programada + hora del día
    df = construir_fecha_hora_programada(df, col_fecha, col_hora, col_destino="fecha_hora_programada")
    df["hora_del_dia"] = df["fecha_hora_programada"].dt.hour

    # 4. Categorización por niveles (bins). Los vuelos cancelados no tienen
    # demora numérica (minutos_netos_demora es NaN), entonces pd.cut los deja
    # en NaN; los marcamos explícitamente como "Cancelado" con una máscara
    # booleana en vez de depender del truco implícito del string "nan".
    es_demora_cancelada = df["minutos_netos_demora"].isna()
    df["nivel_demora"] = pd.cut(
        df["minutos_netos_demora"], bins=BINS_DEMORA, labels=ETIQUETAS_NIVEL_DEMORA
    ).astype(object)
    df.loc[es_demora_cancelada, "nivel_demora"] = "Cancelado"
    df["nivel_demora"] = pd.Categorical(
        df["nivel_demora"], categories=ORDEN_NIVEL_DEMORA, ordered=True
    )

    return df


## Funciones de importación y unificación

In [4]:
def unificar_datasets(datasets, nombre_archivo):
    """
    Concatena una lista de DataFrames y guarda el resultado como archivo
    Parquet. Lanza un error si los datasets no comparten las mismas columnas,
    para evitar generar en silencio un archivo unificado lleno de NaN por un
    desajuste de esquema.
    """
    columnas_por_dataset = [frozenset(d.columns) for d in datasets]
    if len(set(columnas_por_dataset)) > 1:
        raise ValueError(f"Los datasets tienen columnas distintas: {[set(c) for c in columnas_por_dataset]}")

    df_final = pd.concat(datasets, ignore_index=True)
    ruta_salida = f"{nombre_archivo}.parquet"
    df_final.to_parquet(ruta_salida)
    print(f"¡Todo unificado y guardado en '{ruta_salida}'!")
    return df_final


def cargar_y_preparar_aerolinea(file_path, titulo_reporte):
    """Carga un archivo crudo de aerolínea, valida y renombra sus columnas, y corre un reporte rápido."""
    df = cargar_datos(file_path)
    validar_columnas_requeridas(df, COLUMNAS_CRUDAS_REQUERIDAS)
    df = df.rename(columns=MAPEO_COLUMNAS)
    reporte_basico_df(df, titulo_reporte)
    return df


# Punto de entrada del script


In [6]:
def main():
    # --- Importación de archivos ---
    df_jetsmart = cargar_y_preparar_aerolinea(
        "vuelos_anual_WJ_consolidado_2025.parquet", "Jetsmart"
    )
    df_flybondi = cargar_y_preparar_aerolinea(
        "vuelos_anual_FO_consolidado_2025.parquet", "Flybondi"
    )

    # --- Nota sobre valores nulos ---
    # La columna "Hora Real" tiene más de mil valores nulos porque los
    # vuelos cancelados nunca tienen una hora de despegue real.

    # --- Limpieza e ingeniería de features ---
    df_jetsmart_final = procesar_vuelos(df_jetsmart)
    df_flybondi_final = procesar_vuelos(df_flybondi)

    print(df_jetsmart_final.dtypes)
    print(df_flybondi_final.dtypes)

    # --- Unificación y guardado ---
    datasets_limpios = [df_jetsmart_final, df_flybondi_final]
    unificar_datasets(datasets_limpios, "vuelos_historicos_consolidado")

    return df_jetsmart_final, df_flybondi_final


df_jetsmart_final, df_flybondi_final = main()


Reporte:	 Jetsmart
SHAPE:
-------------------------
Columnas: 8	|	Filas: 23675
INFO
-------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23675 entries, 0 to 23674
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Vuelo            23675 non-null  object
 1   Ruta             23675 non-null  object
 2   hora_programada  23675 non-null  object
 3   Hora Real        23334 non-null  object
 4   demora_cruda     23675 non-null  object
 5   fecha            23675 non-null  object
 6   mes              23675 non-null  int64 
 7   empresa          23675 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.4+ MB

+-----+---------+--------+---------+-------+-------+-------+-------+-------+
|     |   count |   mean |     std |   min |   25% |   50% |   75% |   max |
|-----+---------+--------+---------+-------+-------+-------+-------+-------|
| mes |   23675 | 6.8694 | 3.42739 |     1 |     4

Este notebook finaliza con un dataset completo unificado de las dos aerolineas que fueron scrapeadas.